## testing mlp

In [ ]:
import numpy as np
import pytest
import torch
from torch.nn import functional as F

from src.models.torch_mlp import TorchMLP


def _make_model(p_drop):
    torch.manual_seed(0)
    model = TorchMLP(hidden_sizes=(6, 4), p_drop=p_drop)
    model.n_classes_ = 3
    model.layers_ = model._build_layers(5, 3)
    return model


@pytest.mark.parametrize("p_drop", [0.0, 0.5])
def test_manual_backward_matches_autograd(p_drop):
    model = _make_model(p_drop)
    X = torch.randn(8, 5)
    y = torch.tensor([0, 1, 2, 0, 1, 2, 0, 1])

    # 1) gradient tự viết
    model.generator_ = torch.Generator().manual_seed(123)
    with torch.no_grad():
        inputs, masks, logits = model._forward_cache(X, training=True)
        model._backward(inputs, masks, logits, y)
    manual = [(lin.weight.grad.clone(), lin.bias.grad.clone())
              for lin in model.layers_]

    # 2) gradient của autograd (chỉ để đối chiếu), cùng mặt nạ dropout
    for lin in model.layers_:
        lin.weight.grad = None
        lin.bias.grad = None
    model.generator_ = torch.Generator().manual_seed(123)
    with torch.enable_grad():
        _, _, logits = model._forward_cache(X, training=True)
        F.cross_entropy(logits, y).backward()

    for lin, (gw, gb) in zip(model.layers_, manual):
        torch.testing.assert_close(lin.weight.grad, gw, rtol=1e-4, atol=1e-6)
        torch.testing.assert_close(lin.bias.grad, gb, rtol=1e-4, atol=1e-6)


def test_predict_shapes_and_probabilities():
    rng = np.random.default_rng(0)
    X = rng.standard_normal((50, 8))
    y = rng.integers(0, 4, size=50)
    model = TorchMLP(hidden_sizes=(8,), epochs=2, batch_size=16).fit(X, y)

    assert model.predict(X).shape == (50,)
    proba = model.predict_proba(X)
    assert proba.shape == (50, 4)
    np.testing.assert_allclose(proba.sum(axis=1), 1.0, atol=1e-5)


def test_can_learn_simple_problem():
    rng = np.random.default_rng(1)
    X = rng.standard_normal((150, 8))
    y = X[:, :3].argmax(axis=1)
    model = TorchMLP(hidden_sizes=(32,), learning_rate=0.05, momentum=0.9,
                     epochs=100, batch_size=30, seed=0).fit(X, y)
    assert (model.predict(X) == y).mean() > 0.9


def test_each_hyperparameter_changes_training():
    rng = np.random.default_rng(2)
    X = rng.standard_normal((100, 8))
    y = X[:, :3].argmax(axis=1)
    base = dict(hidden_sizes=(16,), learning_rate=0.05, epochs=3,
                batch_size=20, p_drop=0.0, momentum=0.0, seed=0)
    base_loss = TorchMLP(**base).fit(X, y).history_["loss"][-1]

    changes = {"hidden_sizes": (32,), "learning_rate": 0.1, "batch_size": 50,
               "p_drop": 0.3, "momentum": 0.9}
    for name, value in changes.items():
        loss = TorchMLP(**{**base, name: value}).fit(X, y).history_["loss"][-1]
        assert loss != base_loss, f"{name} has no effect on training"


def test_invalid_hyperparameters_raise():
    with pytest.raises(ValueError):
        TorchMLP(p_drop=1.0)
    with pytest.raises(ValueError):
        TorchMLP(momentum=-0.1)
    with pytest.raises(ValueError):
        TorchMLP(hidden_sizes=())

In [ ]:
param_grid = {
    "hidden_sizes": [(64,), (128,), (256, 128), (1024, 256), (5125, 1024, 128)],
    "p_drop": [0.0, 0.2, 0.4],
    "learning_rate": [0.01, 0.05, 0.1],
    "momentum": [0.0, 0.5, 0.9],
    "batch_size": [64, 128, 256],
}